<a href="https://colab.research.google.com/github/gintaras17/v-numeriu-analize/blob/main/ValstybiniaiNumeriai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Duomenų paruošimas.**

Atsisiunčiu duomenis iš GitHub naudojant requests ir paruošiu juos darbui su Spark.

In [ ]:
# imu iš išorinio GitHub url
import pandas as pd
import requests
import io

# sukuriu tuščią sąrašą, kuriame saugosiu DataFrames gautus iš github, sąrašas kursis kaskart naujai, kadangi yra prieš url paėmimą
dfs = []

# Github nuorodos
urls = [
    "https://raw.githubusercontent.com/marukqs/techmuge/refs/heads/main/valstybiniai_numeriai_csv_part1.csv",
    "https://raw.githubusercontent.com/marukqs/techmuge/refs/heads/main/valstybiniai_numeriai_csv_part2.csv",
    "https://raw.githubusercontent.com/marukqs/techmuge/refs/heads/main/valstybiniai_numeriai_csv_part3.csv"
]

# nuskaitau turinį kaip teksta naudojant requests biblioteką ir for ciklą
for url in urls:
  response = requests.get(url)
  # pasitikrinu ar sujungimas įvyko
  if response.status_code == 200:
  # paverčiu tekstą į pandas DataFrame
  # io.StringIO leidžia read_csv funkcijai nuskaityti teksta lyg tai butu failas
  # naudojant requests ir stulpeliui suteikiu "numeris" pavadinimą
    temp_df = pd.read_csv(io.StringIO(response.text), header=None, names=["numeris"])
  # įdedu į sąrašą
    dfs.append(temp_df)
  else:
    print(f"Klaida siunčiant failą iš: {url}")

# tikrinu sąrašo elementų kiekį prieš išpakuojant
print(f"Sąrašo elementų kiekis: {len(dfs)}")

if len(dfs) == 3:
  # priskiriu kintamiesiems
  df1, df2, df3 = dfs[0], dfs[1], dfs[2]
  print(f"Visi 3 dataframes sukurti sėkmingai. Visas eilučių kiekis: {len(df1) + len(df2) + len(df3)}")
else:
  print("Kažkas nepavyko, rasta mažiau nei 3 failai")

Pakuriu Spark sesiją, sukuriu dar vieną stulpelį ir suteikiu jam pavadinimą bei statinį pavadinimą visiems duomenims ir viską sujungiu į viena Spark DataFrame.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit

# sukuriu spark sesija, pagrindinį įrankį duomenų apdorojimui ir suteikiu pavadinimą
spark = SparkSession.builder.appName("duomenys").getOrCreate()

# pridedu stulpelį "failo_vardas" naudodamas .withColumn() ir su lit() pridedu statinį tekstą to stulpelio duomenims
sdf1 = spark.createDataFrame(df1).withColumn("failo_vardas", lit("part1.csv"))
sdf2 = spark.createDataFrame(df2).withColumn("failo_vardas", lit("part2.csv"))
sdf3 = spark.createDataFrame(df3).withColumn("failo_vardas", lit("part3.csv"))

# sujungiu visus DataFrames į vieną
sdf_all = sdf1.union(sdf2).union(sdf3)
# padedu į atmintį, kad vėliau procesai veiktų greičiau
sdf_all.cache()
# patikrinu eilučių skaičių
print(f"Iš viso eilučių po sujungimo: {sdf_all.count()}")

In [ ]:
# Surūšiuoju atsitiktine tvarka, kad pažiūrėčiau kaip atrodo
from pyspark.sql.functions import rand
sdf_all.orderBy(rand()).show()

In [ ]:
# skaičiuoju pasikartojančius numerius ir pasižiūriu pasikartojančius numerius mažėjimo tvarka
duplicates = sdf_all.groupBy("numeris").count() #groupBy sugrupuoja, .count() suskaičiuoja kievkienos grupės dydį
# duplicates.show()
duplicates.filter("count > 1").orderBy("count", ascending=False).show(20)

In [ ]:
# pašalinu dublikatus
sdf_unique = sdf_all.dropDuplicates(["numeris"])

# skaičiuoju pasikartojančių kiekį
dublicates_qty = sdf_all.count() - sdf_unique.count()
print(f"Dublikatų rasta: {dublicates_qty}")

# patikrinu rezultatą prieš ir po
print("Prieš dublikatų pašalinimą:", sdf_all.count())
print("Po dublikatų pašalinimo:", sdf_unique.count())

Klasifikacija.

Naudoju rlike funkciją su Regex, kad priskirčiau numerius atitinkamoms transporto priemonių grupėms.

In [ ]:
# klasifikuoju numerius pagal jų tipą, ir priskiriu kuris kam priklausytų
from pyspark.sql.functions.builtin import col, when, desc

# su .withColumn("naujas_stulpelis", funkcijos) pridedu naują stulpelį "tipas", kuriame susidės numerių tipai
sdf_classified = sdf_unique.withColumn(
    "tipas",
    # filtruoju naudodamas r.like() regex taip, kad nepasimestų nei vienas iš numerių (DI padėjo atrasti optimaliausią variantą)
    when(col("numeris").rlike("^EV[0-9]{4}$"), "elektromobilių")
    # pirmas when eina be taško. when atlieka tokią pačią funkciją kaip būtų su if, else,...
    .when(col("numeris").rlike("^[0-9]{5,6}$"), "diplomatinių")
    .when(col("numeris").rlike("^[A-Z]{3}[0-9]{3}$"), "bendro naudojimo")
    .when(col("numeris").rlike("^[0-9]{4}[A-Z]{1}$"), "istorinių motociklų/mopedų")
    .when(col("numeris").rlike("^[0-9]{3}[A-Z]{2}$"), "motociklų")
    .when(col("numeris").rlike("^[0-9]{2}[A-Z]{3}$"), "mopedų")
    .when(col("numeris").rlike("^[A-Z]{2}[0-9]{2}$"), "keturračių")
    .when(col("numeris").rlike("^[A-Z]{2}[0-9]{3}$"), "priekabų/puspriekabių")
    # Vardiniai numeriai: nuo 1 iki 6 simbolių, leidžiant skaičius ir LT raides,
    # bet privaloma bent viena raidė (naudojamas lookahead užtikrinimui).
    .when(col("numeris").rlike("^(?=.*[A-ZĄČĘĖĮŠŲŪŽ])[A-Z0-9ĄČĘĖĮŠŲŪŽ]{1,6}$"), "vardinių")
    .otherwise("kita")
)

sdf_classified.groupBy("tipas").count().orderBy(desc("count")).show(12)

In [ ]:
from pyspark.sql import functions as F

# Sukuriu papildomą klasifikaciją su klaidų identifikavimu
sdf_analyzed = sdf_unique.withColumn(
    "komentaras",
    # KLAIDA: Per trumpas arba per ilgas
    F.when((F.length("numeris") < 3) | (F.length("numeris") > 6), "KLAIDA: Netinkamas ilgis")

    # KLAIDA: Turi specialiųjų simbolių (ne raidės ir ne skaičiai)
    .when(F.col("numeris").rlike("[^A-Z0-9ĄČĘĖĮŠŲŪŽ]"), "KLAIDA: Draudžiami simboliai")

    # KLAIDA: Vardinis numeris sudarytas TIK iš skaičių (Regitros taisyklė)
    # Jei tai ne diplomatinis 5-6 skaitmenų numeris
    .when(F.col("numeris").rlike("^[0-9]+$") & ~F.col("numeris").rlike("^[0-9]{5,6}$"), "KLAIDA: Tik skaičiai vardiniame")

    # Jei atitinka vardinio formatą (bent viena raidė, 3-6 simb.)
    .when(F.col("numeris").rlike("^[A-Z0-9ĄČĘĖĮŠŲŪŽ]{3,6}$"), "Teisingas: Vardinis")

    .otherwise("KLAIDA: Neatpažintas formatas")
)

# Išfiltruoju tik klaidas sąrašui pateikti
bad_plates = sdf_analyzed.filter(F.col("komentaras").contains("KLAIDA"))

# Parodau rezultatus
print("Neteisingų numerių analizė:")
bad_plates.select("numeris", "komentaras").show(truncate=False)

In [ ]:
# pridedu naują stulpelį "eil_nr" ir suteikiu eilės nr su window funkcija
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# pasak DI tai paprasciausias variantas visai lentelei
window_sort = Window.orderBy("numeris")

# stulpeliams suteikiu eiles nr is eiles nuo 1 iki ...
sdf_final = sdf_classified.withColumn(
    "eil_nr",
    row_number().over(window_sort)  #row_number() generuoja 1, 2, 3, 4... .over(window_sort) pasako pagal kokia logika numeruoti
    )

# pasitikrinu kas gavosi
sdf_final.show(truncate=False)

In [ ]:
from os import truncate
from pyspark.sql.functions import count # naudojant agregavimo funkcijas skaiciuojiu kiek yra kiekvieno tipo numeriu
from pyspark.sql import functions as F

sdf_analize = sdf_classified.groupBy("tipas").agg(
    F.count("numeris").alias("kiekis")
).sort(F.desc("kiekis"))

sdf_analize.show(truncate=False)

In [ ]:
from pyspark.sql import functions as F

# Išfiltruoju visus numerius, kurie gavo bet kokią "KLAIDA" žymę ir neatitinka regitros taisyklių
list_of_bad = sdf_analyzed.filter(F.col("komentaras").contains("KLAIDA"))

# Suskaičiuoju, kiek iš viso klaidų rasta
qty_of_bad = list_of_bad.count()

# Pateikiu rezultatą
print(f"Iš viso rasta neteisingų numerių: {qty_of_bad}")

if qty_of_bad > 0:
    print("Neteisingų numerių pavyzdžiai ir jų problemos:")
    list_of_bad.select("numeris", "komentaras").show(truncate=False)
else:
    print("Visi numeriai atitinka taisykles!")


Window funkcijos.

Naudoju Window.partitionBy, kad kiekvienoje kategorijoje rasčiau 5 populiariausias pradžios raides.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Atlieku papildomą analizę, ieškau 5 dažniausiai pasikartojančių pirmų raidžių naudojant window funkciją
df_with_char = sdf_classified.withColumn("pirma_raide", F.substring(F.col("numeris"), 1, 1))

# Suskaičiuoju dažnumą: kiekvienam tipui ir kiekvienai raidei
counts_df = df_with_char.groupBy("tipas", "pirma_raide").agg(F.count("*").alias("kiekis"))

# Sukuriu Window funkciją: skaidau (partition) pagal tipą, rikiuoju pagal kiekį
window_spec = Window.partitionBy("tipas").orderBy(F.desc("kiekis"))

# Suteikiu reitingą (row_number) ir palieku tik TOP 5
top_5_all = counts_df.withColumn("reitingas", F.row_number().over(window_spec)).filter(F.col("reitingas") <= 5)

# Išskirstau į atskirus DataFrame
# Sukuriu žodyną (dictionary), kur raktas - tipo pavadinimas, o reikšmė - jo TOP 5 DataFrame
categories = [row.tipas for row in top_5_all.select("tipas").distinct().collect()]
separate_df = {cat: top_5_all.filter(F.col("tipas") == cat).drop("reitingas") for cat in categories}

# # kaip pasiekti konkretų DataFrame
# print("Bendro naudojimo numerių TOP 5 raidės:")
# separate_df['bendro naudojimo'].show()

# Rodau 50 eilučių, kad tilptų visų kategorijų TOP 5
top_5_all.orderBy("tipas", F.desc("kiekis")).show(50, truncate=False)

*   Iš viso apdorota beveik 8 mln. įrašų.
*   Rasta ~860 tūkst. pasikartojančių, kurie buvo sėkmingai pašalinti.
*   Klasifikacija parodė, kad dominuoja bendro naudojimo numeriai.
*   Tik 1 numeris neatitiko jokių nustatytų taisyklių.